# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamyaMuthumariappan/FlyRankAI/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question.** Which pages should a content editor open first this week, and does a model's apparent skill at ranking them survive an honest audit? Decision-support for a human editor; the ranking orders attention and never acts on a page.

Detail: paper section 1, and `w01_research_question.ipynb`, `w02_ml_task_framing.ipynb`.

In [1]:
# 1. Question -> the proxy label, written as code, and its base rate.
# The label is a RULE over ordinary columns (no editor-confirmed outcome exists), so scores measure
# agreement with this rule. trend_direction/trend_pct define "declining" and are never features.
def proxy_high(avg_position, is_declining, days_since_update, impressions_30d):
    page1_decay   = (0 < avg_position <= 10) and is_declining          # needs the decline signal
    stale_visible = days_since_update > 60 and impressions_30d > 50    # readable without any trend
    return int(page1_decay or stale_visible)

N_PAGES, N_HIGH = 30_000, 12_694            # receipts from w02/w05 (printed there)
print(f"proxy base rate = {N_HIGH:,} / {N_PAGES:,} = {N_HIGH/N_PAGES:.3f}")
print("every precision@K or AUC below must be read against this base rate")

proxy base rate = 12,694 / 30,000 = 0.423
every precision@K or AUC below must be read against this base rate


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Data.** 30,000-page anonymized snapshot (32 clients, trailing 90 days) for the baseline, model and playbook. The warehouse release (build v20260703, 78,835,655 daily rows) was used for the data contract on the March 2026 partition only (9,841,378 daily rows, 331,437 page-month rows); June 2026 stayed sealed. No model was trained on the full warehouse. Label-derived columns (`trend_direction`, `trend_pct`, `is_declining_label`) and pseudonymous IDs are never features.

Detail: paper section 2, and `w03_data_contract.ipynb`.

In [2]:
# 2. Data -> what was used, as a receipts table (numbers printed in w03 and the starter data dictionary).
import pandas as pd
data_facts = pd.DataFrame([
    ["Modelling snapshot (baseline, model, playbook)", "30,000 pages x 44 columns, 32 clients, trailing 90 days"],
    ["Warehouse release (build v20260703)",             "78,835,655 daily rows; 2025-01-27 to 2026-06-30"],
    ["Used for the data contract only",                 "March 2026 partition: 9,841,378 daily rows, 55 clients"],
    ["Aggregated grain checked",                        "331,437 page-month rows, 0 duplicate (client, page) pairs"],
    ["Sealed",                                          "June 2026 final month, never touched"],
    ["GA4 flag confirmed TRUE (March row-days)",        f"{413_966/9_841_378:.1%}  (413,966 of 9,841,378)"],
    ["Position == 0 means no data",                     f"{1_205/30_000:.1%}  (1,205 pages) routed to a data check"],
], columns=["what", "value"])
print("No model in this project was trained on the full warehouse.\n")
data_facts

No model in this project was trained on the full warehouse.



,what,value
0,"Modelling snapshot (baseline, model, playbook)","30,000 pages x 44 columns, 32 clients, trailin..."
1,Warehouse release (build v20260703),"78,835,655 daily rows; 2025-01-27 to 2026-06-30"
2,Used for the data contract only,"March 2026 partition: 9,841,378 daily rows, 55..."
3,Aggregated grain checked,"331,437 page-month rows, 0 duplicate (client, ..."
4,Sealed,"June 2026 final month, never touched"
5,GA4 flag confirmed TRUE (March row-days),"4.2% (413,966 of 9,841,378)"
6,Position == 0 means no data,"4.0% (1,205 pages) routed to a data check"


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Methodology.** Proxy label `y_high` = page-one decay OR stale-but-visible (base rate 0.423). Rule baseline with reason codes; logistic regression and Random Forest on 28 features; client-grouped splits (75/25 and 5-fold out-of-fold); leakage checks: banned-column guard, planted-leak test, ingredient audit, ablation, permutation importance. Seed 42.

Detail: paper section 3, and `w04_baseline_score.ipynb`, `w05_model.ipynb`, `w06_validation_audit.ipynb`.

In [3]:
# 3. Methodology -> the transparent baseline as real, runnable code, shown on 3 TOY rows.
# The toy rows below are invented only to show the rule's mechanics. They are not data.
import math
def baseline_score(avg_position, ctr_pct, days_since_update, impressions_30d):
    visible      = impressions_30d >= 50
    stale_flag   = days_since_update >= 90
    ctr_gap_flag = (0 < avg_position <= 20) and (ctr_pct < 0.35)
    base = math.log1p(impressions_30d) * visible
    score = 0.6 * ctr_gap_flag * base + 0.4 * stale_flag * base
    if ctr_gap_flag and visible:   code = "low_ctr_visible_page"
    elif stale_flag and visible:   code = "stale_visible_page"
    else:                          code = "general_monitor"
    return round(score, 3), code

toy = pd.DataFrame(
    [[2.5, 0.15, 20, 89_000], [27.9, 0.21, 104, 140_000], [8.5, 0.80, 20, 600]],
    columns=["avg_position", "ctr_pct", "days_since_update", "impressions_30d"])
toy[["score", "reason_code"]] = [baseline_score(*r) for r in toy.itertuples(index=False)]
print("Models: logistic regression + Random Forest, 28 features (20 numeric, 8 categorical), seed 42.")
print("Splits: one client-grouped 75/25 (24 train / 8 test clients) + client-grouped 5-fold out-of-fold.")
print("Leakage checks: banned-column guard, planted leak, ingredient audit, ablation, permutation importance.\n")
toy

Models: logistic regression + Random Forest, 28 features (20 numeric, 8 categorical), seed 42.
Splits: one client-grouped 75/25 (24 train / 8 test clients) + client-grouped 5-fold out-of-fold.
Leakage checks: banned-column guard, planted leak, ingredient audit, ablation, permutation importance.



,avg_position,ctr_pct,days_since_update,impressions_30d,score,reason_code
0,2.5,0.15,20,89000,6.838,low_ctr_visible_page
1,27.9,0.21,104,140000,4.74,stale_visible_page
2,8.5,0.80,20,600,0.0,general_monitor


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Results.** Same 8-client test split (base rate 0.364): precision@1,000 is 0.463 (baseline), 0.776 (logistic regression), 0.903 (Random Forest). Client-grouped 5-fold AUC 0.946 (random 5-fold 0.960); removing four label-ingredient columns drops it to 0.658; a planted `trend_pct` leak gives 0.998. CTR-gap pages show a decline-proxy rate of 0.614 against 0.542 overall.

Detail: paper section 4, and `w05_model.ipynb`, `w06_validation_audit.ipynb`, `w07_action_playbook.ipynb`.

In [4]:
# 4. Results -> model vs baseline on the SAME held-out clients (receipts printed in w05 and w06).
TEST_BASE = 0.364
pk = pd.DataFrame({
    "K":                   [30, 50, 100, 300, 500, 1000, 2000, 3000],
    "rule baseline":       [1.000, 0.980, 0.910, 0.653, 0.550, 0.463, 0.492, 0.475],
    "logistic regression": [1.000, 1.000, 1.000, 0.997, 0.860, 0.776, 0.746, 0.677],
    "random forest":       [1.000, 1.000, 1.000, 1.000, 1.000, 0.903, 0.782, 0.720],
}).set_index("K")
print(f"precision@K, 8 held-out clients, 7,115 pages, base rate {TEST_BASE}")
print(pk.to_string())
print("\nlift over base rate at K=1,000:", {c: round(float(pk.loc[1000, c]) / TEST_BASE, 2) for c in pk.columns})

ablation = pd.DataFrame([
    ["A. all 28 features",                          0.946, 0.915],
    ["B. minus 4 label-ingredient columns",         0.658, 0.567],
    ["C. B minus 4 volume-proxy columns",           0.586, 0.475],
    ["planted leak: A + trend_pct (harness check)", 0.998, 0.994],
], columns=["client-grouped 5-fold, out-of-fold", "AUC", "worst fold"])
print("\n" + ablation.to_string(index=False))

precision@K, 8 held-out clients, 7,115 pages, base rate 0.364
      rule baseline  logistic regression  random forest
K                                                      
30            1.000                1.000          1.000
50            0.980                1.000          1.000
100           0.910                1.000          1.000
300           0.653                0.997          1.000
500           0.550                0.860          1.000
1000          0.463                0.776          0.903
2000          0.492                0.746          0.782
3000          0.475                0.677          0.720

lift over base rate at K=1,000: {'rule baseline': 1.27, 'logistic regression': 2.13, 'random forest': 2.48}

         client-grouped 5-fold, out-of-fold   AUC  worst fold
                         A. all 28 features 0.946       0.915
        B. minus 4 label-ingredient columns 0.658       0.567
          C. B minus 4 volume-proxy columns 0.586       0.475
planted leak: A + tr

## 5. Limitations

*What this work cannot claim.*

**Limitations.** Proxy target; the strongest features are ingredients of that proxy; one snapshot with no time split; no causal claim about refreshing; population filter; large-client dominance; uneven client coverage; assumed (not measured) effort figures.

Detail: paper section 5, and `w06_validation_audit.ipynb` section 4.

In [5]:
# 5. Limitations -> the arithmetic behind the limits, so the sentences cannot drift from the numbers.
auc_all, auc_no_ingredients = 0.946, 0.658
print(f"AUC drop from removing 4 label-ingredient columns: {auc_all - auc_no_ingredients:.3f}")
print(f"random 5-fold (0.960) minus client-grouped (0.946): {0.960 - 0.946:.3f}  -> client memorising is a small factor")

true_high_test, K = 2_591, 1_000
print(f"\nW5 test set: {true_high_test:,} true highs, ranking K={K:,} -> at least {true_high_test - K:,} misses are")
print("unavoidable by arithmetic, so the false-negative surplus says little about model caution")

share_stale_only = 0.557
print(f"\n{share_stale_only:.0%} of proxy-high pages are high through staleness+visibility alone (no decline signal needed)")
print("Not claimed: any causal effect of refreshing; any time-aware result (single snapshot, no dated rows).")

AUC drop from removing 4 label-ingredient columns: 0.288
random 5-fold (0.960) minus client-grouped (0.946): 0.014  -> client memorising is a small factor

W5 test set: 2,591 true highs, ranking K=1,000 -> at least 1,591 misses are
unavoidable by arithmetic, so the false-negative surplus says little about model caution

56% of proxy-high pages are high through staleness+visibility alone (no decline signal needed)
Not claimed: any causal effect of refreshing; any time-aware result (single snapshot, no dated rows).


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**Ranked recommendations.** 1) CTR-gap lane first, with a results-page check; 2) small frequent review batches; 3) treat stale-but-visible as a question; 4) work the queue per client; 5) model as a second opinion only; 6) fix missing data and require a second reviewer for very high-volume pages; 7) build the decision log and an untouched holdout so the next study is measurable. Never automate publishing or pruning.

Detail: paper section 6, and `w07_action_playbook.ipynb`.

In [6]:
# 6. Ranked recommendations -> the ordered playbook, plus the value-concentration receipts behind "small batches".
recs = pd.DataFrame([
    [1, "Work the CTR-gap lane first (8,780 pages)",          "decline proxy 0.614 vs 0.542 overall"],
    [2, "Review in small, frequent batches",                  "first 250 pages hold 29% of the clicks ceiling"],
    [3, "Treat stale-but-visible as a question, not a verdict","staleness test was MIXED; 68% of pages share two update dates"],
    [4, "Work the queue per client",                          "largest client holds 50% of the global top 50"],
    [5, "Model as a second opinion only",                     "AUC 0.946 -> 0.658 without 4 label-ingredient columns"],
    [6, "Fix missing data; second reviewer for 50k+ impressions", "1,205 pages have no position"],
    [7, "Build a decision log + untouched ~10% holdout",      "without it, 'did actioned pages improve?' cannot be measured"],
], columns=["rank", "action", "evidence"])
conc = pd.DataFrame({"pages reviewed": [50, 250, 1000, 12708],
                     "share of ceiling (queue order)": [0.123, 0.292, 0.548, 1.0],
                     "share of ceiling (best case)":   [0.145, 0.339, 0.604, 1.0]})
print(conc.to_string(index=False), "\n(ceiling = 151,433 clicks / 90 days: an upper bound, not a forecast)\n")
print(recs.to_string(index=False))

 pages reviewed  share of ceiling (queue order)  share of ceiling (best case)
             50                           0.123                         0.145
            250                           0.292                         0.339
           1000                           0.548                         0.604
          12708                           1.000                         1.000 
(ceiling = 151,433 clicks / 90 days: an upper bound, not a forecast)

 rank                                                 action                                                      evidence
    1              Work the CTR-gap lane first (8,780 pages)                          decline proxy 0.614 vs 0.542 overall
    2                      Review in small, frequent batches                first 250 pages hold 29% of the clicks ceiling
    3   Treat stale-but-visible as a question, not a verdict staleness test was MIXED; 68% of pages share two update dates
    4                              Work the que

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

**Artifacts.** The deployed paper is the single static page `docs/index.html`, served by GitHub Pages (Settings → Pages → Deploy from branch `main`, folder `/docs`). Every figure value in it is copied from a printed notebook output. Live paper: https://ramyamuthumariappan.github.io/FlyRankAI/

In [7]:
# 7. Artifacts -> check the deployed paper file (docs/index.html) has every required section and the data credit.
import pathlib
root = pathlib.Path.cwd()
while not (root / "docs" / "index.html").exists() and root != root.parent:
    root = root.parent
html = (root / "docs" / "index.html").read_text(encoding="utf-8")
sections = ["abstract","intro","data","method","results","limits","recs","repro","credit"]
print("sections present:", {s: f'id="{s}"' in html for s in sections})
print("data credit linked:", "https://flyrank.ai" in html)
print("capstone notebook linked:", "capstone.ipynb" in html)
print("paper_url.txt:", (root / "submission" / "paper_url.txt").read_text().strip())


sections present: {'abstract': True, 'intro': True, 'data': True, 'method': True, 'results': True, 'limits': True, 'recs': True, 'repro': True, 'credit': True}
data credit linked: True
capstone notebook linked: True
paper_url.txt: https://ramyamuthumariappan.github.io/FlyRankAI/


## 8. ML-12 — demo, social post, employer summary

**5-minute demo outline**
1. (0:00) The decision: an editor has time to open a few pages this week. Which ones first? (paper §1)
2. (0:45) The data: 30,000 anonymized pages from 32 clients; the warehouse was used only to check the data contract on one March 2026 partition. No client names, URLs or queries. (§2)
3. (1:30) The method: a readable rule baseline with reason codes, then logistic regression and Random Forest, validated by holding out whole clients. (§3)
4. (2:15) The result: on the same 8 held-out clients, precision@1,000 is 0.463 (rule), 0.776 (logistic), 0.903 (Random Forest), against a 0.364 base rate. (§4)
5. (3:15) The honest part: AUC falls from 0.946 to 0.658 when four columns that define the label are removed, and a planted leak scores 0.998, so the audit works and the model mostly re-reads the rule. (§4, §5)
6. (4:15) The playbook: CTR-gap lane first, small review batches, model as a second opinion only, never automate publishing or pruning. (§6)
7. (4:45) What I would do next: a decision log and an untouched holdout so a time-aware, outcome-based study becomes possible.

**Social-post cut**
I finished my FlyRank ML internship capstone: a client-grouped audit of a "which page should an editor review first?" ranking. The model looked great (AUC 0.946) until I removed the four columns that define the label, and it dropped to 0.658. Writing down what a model only re-reads, and what it actually learns, was the most useful part. Paper and code: https://ramyamuthumariappan.github.io/FlyRankAI/ #MachineLearning #SEO #DataScience

**Employer-facing summary (3 sentences)**
I built and audited a page-review ranking on anonymized FlyRank search data, comparing a transparent rule baseline with logistic regression and a Random Forest using client-grouped validation. My leakage audit showed that most of the apparent skill came from columns that define the label, so I framed the output as decision-support with reason codes rather than a forecast. The result is a deployed research paper, a reproducible notebook trail, and a ranked playbook for human editors.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
